# FeO aerosols in rock-vapour atmospheres

How does mantle redox state affect the rock-vapour atmosphere above a magma ocean? When does wüstite (FeO) condense into aerosols, and what do those aerosols do to the emission and reflection spectra?

We sweep the oxygen fugacity from reducing (IW−4) to oxidising (IW+2), relative to the iron–wüstite buffer of O'Neill & Eggins (2002). For each case:

1. **Melt composition.** The FeO content of the silicate melt is taken from ideal equilibrium with a metallic iron core at that ΔIW, starting from a BSE composition and capped at `FEO_MAX` (see below).
2. **Vapour composition.** We use **LavAtmos** (Van Buchem et al. 2023, [arXiv:2210.10463](https://arxiv.org/abs/2210.10463)) at the imposed fO$_2$ and the melt temperature. We do *not* use LavAtmos's congruent-vaporisation mass balance, which would fix fO$_2$ at about IW+2.4 whatever the mantle is doing. Instead, the mantle acts as an oxygen buffer, and the notebook records the oxygen it would have to take up or supply.
3. **Atmospheric structure.** The surface partial pressures give the surface pressure and the well-mixed VMRs, which are passed to **AGNI** through `juliacall`. We use the `Volenfell/128` spectral file (128 bands) and a fixed surface temperature (`sol_type=1`). Each case is run in three stages:
    * `base`: no condensation, used as a reference.
    * `cond`: FeO rainout (condensation plus cold trap), with aerosol opacity off.
    * `aer`: as `cond`, plus Mie-scattering wüstite aerosols whose mass is set by the FeO condensate yield.

**Solver mode.** `SOLVER = "prescribed"` imposes the T(p) structure (dry adiabat plus an isothermal stratosphere at the skin temperature) with `solve_prescribed`. Condensation, aerosols and fluxes are then evaluated on that fixed structure. This is fast, and it shows where FeO *can* condense, but the result is not a radiative–convective equilibrium. `SOLVER = "energy"` solves for radiative–convective equilibrium with `solve_energy`, running `base` → `cond` → `aer` as a continuation, each stage starting from the previous solution. Latent heat is excluded throughout (`latent=false`).

**Environment.** Use a Python kernel with `thermoengine` (MELTS), `juliacall`, `calliope`, `netCDF4`, `pandas` and `matplotlib` (e.g. the `prt` conda env). `RAD_DIR` must point to a compiled SOCRATES, and a compiled FastChem is required.

In [ ]:
import os, sys, time, logging, contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import netCDF4 as nc
from scipy.optimize import brentq
from calliope.oxygen_fugacity import OxygenFugacity

## Configuration

* The planet is a generic Earth-sized lava world, with the surface held at the melt temperature.
* The instellation is `INST_FACTOR` × $\sigma T_\mathrm{s}^4/(s_0\cos\theta)$, with $s_0$ and $\theta$ the AGNI default flux scale factor and zenith angle. A factor of 1 would let a transparent, zero-albedo atmosphere absorb as much stellar flux as the surface emits. We use a lower value so that the upper atmosphere is cool enough for FeO to condense. In `prescribed` mode, the stratosphere temperature is the skin temperature for this instellation.
* Every figure is also saved as a PDF in `OUT_DIR`.

In [ ]:
# Paths
AGNI_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))   # notebook lives in misc/science/
LAVA_DIR = os.path.join(AGNI_DIR, "LavAtmos")
FC_DIR   = os.path.join(AGNI_DIR, "fastchem")
OUT_DIR  = os.path.join(AGNI_DIR, "out", "feo_aerosols")
os.makedirs(OUT_DIR, exist_ok=True)
assert "RAD_DIR" in os.environ, "RAD_DIR must point to a compiled SOCRATES installation"

def save_fig(fig, name):
    # Save each figure as a PDF in OUT_DIR
    fig.savefig(os.path.join(OUT_DIR, name + ".pdf"), bbox_inches="tight")

SPFILE    = os.path.join(AGNI_DIR, "res", "spectral_files", "Volenfell", "128", "Volenfell.sf")
STAR      = os.path.join(AGNI_DIR, "res", "stellar_spectra", "sun.txt")
BASE_MELT = os.path.join(LAVA_DIR, "input", "lava_compositions", "BSE_palm.csv")   # BSE, FeO = 8.1 wt%

# Redox sweep
T_MELT  = 2500.0                          # surface = melt temperature [K]
DIW     = [-4, -3, -2, -1, 0, 1, 2]       # log10 fO2 relative to IW (O'Neill & Eggins 2002)
FEO_MAX = 18.0                            # [wt%] cap on melt FeO, see "Melt compositions"

# Planet and star
SIGMA    = 5.670374419e-8
RADIUS   = 6.371e6        # [m]
GRAVITY  = 9.81           # [m s-2]
S0_FACT  = 0.6652         # AGNI default flux scale factor
ZENITH   = 60.0           # AGNI default zenith angle [deg]
ALBEDO_B = 0.0            # no pseudo-albedo; reflection is computed by the RT
INST_FACTOR  = 0.3
INSTELLATION = INST_FACTOR * SIGMA * T_MELT**4 / (S0_FACT * np.cos(np.radians(ZENITH)))   # [W m-2]

# AGNI grid and solver
SOLVER    = "prescribed"  # "prescribed" (solve_prescribed, fast) or "energy" (solve_energy, RCE)
ATM_TYPE  = 3             # prescribed structure: 1 isothermal, 2 dry adiabat, 3 dry adiabat + stratosphere
NLEV      = 40
P_TOP     = 1e-8          # [bar]
MAX_STEPS = 200
VMR_FLOOR = 1e-10         # drop gases below this VMR before passing them to AGNI

# FeO aerosol: wüstite optical constants (res/refractive/FeO.txt), log-normal size distribution
AEROSOL = dict(method="mie", species="FeO", nk_file="FeO", r_eff=0.5e-6, sigma_g=1.65)

T_skin = (INSTELLATION / (4 * SIGMA)) ** 0.25 * 0.5 ** 0.25    # as in AGNI phys.calc_Tskin, zero albedo
print(f"Instellation = {INSTELLATION:.3e} W m-2 ({INSTELLATION/1361.0:.0f} x solar constant), T_skin = {T_skin:.0f} K")

## LavAtmos setup

LavAtmos needs a `paths` object, which it does not define itself. The class below follows the `paths_importer` in PROTEUS (`src/proteus/outgas/lavatmos.py`), with the same attribute names.

The function `vaporise_fixed_fO2` goes through the same steps as `melt_vapor_system.vaporise()`, with one difference: fO$_2$ is imposed rather than solved for. The steps are:

1. The MELTS melt activities and the equilibrium constants of the vaporisation reactions are computed at $T_\mathrm{melt}$.
2. LavAtmos's inner solve finds the gas-phase oxygen abundance that reproduces the imposed fO$_2$ in FastChem.
3. The vapour speciation at that fO$_2$ is returned.

`mass_balance_equation_fastchem` also returns the congruent-vaporisation residual: oxygen released by the vapour minus oxygen demanded by the metal cations, weighted per species and in bar. A non-zero residual is the oxygen that the buffering mantle takes up or supplies. The residual is zero where the melt vaporises congruently, which for BSE at 2500 K is at about IW+2.4.

In [ ]:
os.environ["LAVA_DIR"] = LAVA_DIR
os.environ["FC_DIR"]   = FC_DIR
if LAVA_DIR not in sys.path:
    sys.path.insert(0, LAVA_DIR)
import lavatmos3

class LavaPaths:
    def __init__(self, out):
        self.lavatmos_dir  = LAVA_DIR + "/"
        self.input_dir     = os.path.join(LAVA_DIR, "input") + "/"
        self.fastchem3_dir = FC_DIR + "/"
        self.fastchem3_config_template  = os.path.join(self.input_dir, "fastchem3", "config_template.input")
        self.element_abundance_template = os.path.join(self.input_dir, "fastchem3", "element_abundances",
                                                       "element_abundances_template2.dat")
        self.species_data_file      = os.path.join(FC_DIR, "input", "logK", "logK.dat")
        self.species_data_file_cond = os.path.join(FC_DIR, "input", "logK", "logK_condensates.dat")
        os.makedirs(os.path.join(out, "element_abundances"), exist_ok=True)
        os.makedirs(os.path.join(out, "fastchem"), exist_ok=True)
        self.element_abundance_output = os.path.join(out, "element_abundances", "element_abundances_output.dat")
        self.fastchem3_output = os.path.join(out, "fastchem", "")    # LavAtmos appends file names to this
        self.output_dir       = self.fastchem3_output
        self.janafdata        = os.path.join(LAVA_DIR, "data")

@contextlib.contextmanager
def in_dir(path):
    # LavAtmos resolves some data paths relative to the working directory
    old = os.getcwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(old)

VOLATILES  = {"C": 1e-20, "H": 1e-9, "N": 1e-20, "S": 1e-20, "P": 1e-20, "O": 1e-20}   # trace only
P_VOLATILE = 1e-8      # [bar]
P_MELT     = 0.01      # [bar], pressure for the MELTS activity calculation

def vaporise_fixed_fO2(system, T, melt_comp, fO2):
    # Vapour above the melt at imposed fO2 [bar]; returns (FastChem partial pressures [bar], O mass-balance residual)
    Ta = np.array([T])
    system.P_volatile = P_VOLATILE
    system.melt_comp = system.melt.set_melt_comp(melt_comp, False)
    system.melt.calculate_melt_chemical_potentials(Ta, P_MELT, system.thermo_data)
    system.melt.calculate_melt_activities(Ta, P_MELT)
    system.logKr = system.logKr_calc(Ta, system.melt.mu0_liquid)
    resid = system.mass_balance_equation_fastchem(fO2, Ta, VOLATILES)      # also sets system.O_abun
    vpp = system.vapor_partial_pressure_calc(fO2, Ta)
    P_boa = vpp.sum(axis=1).iloc[0] + fO2 + P_VOLATILE
    pp = system.calculate_partial_pressures_fastchem_loop([system.O_abun], Ta, [P_boa], vpp, VOLATILES, meltfrac=1.0)
    return pp.iloc[0], float(resid)

## Melt compositions

Starting from BSE, we set FeO and rescale the other oxides so that their relative proportions and the total are unchanged. All iron is ferrous; there is no Fe$_2$O$_3$.

The FeO content for each ΔIW follows from equilibrium between the silicate and a pure iron core, via the reaction $\mathrm{Fe} + \tfrac{1}{2}\mathrm{O_2} = \mathrm{FeO}$:

$$\Delta\mathrm{IW} = 2\log_{10}\!\left(\frac{x_\mathrm{FeO}^\mathrm{sil}\,\gamma_\mathrm{FeO}}{x_\mathrm{Fe}^\mathrm{met}\,\gamma_\mathrm{Fe}}\right) \approx 2\log_{10} x_\mathrm{FeO}^\mathrm{sil}.$$

This takes ideal mixing and pure iron metal, so $x_\mathrm{FeO} = 10^{\Delta\mathrm{IW}/2}$, with $x_\mathrm{FeO}$ the oxide mole fraction. **It is an order-of-magnitude guide only.**

The relation cannot reach ΔIW > 0 at all while metal is present, since $x_\mathrm{FeO} \le 1$. It also gives about 39 wt% FeO already at IW−1. MELTS-based activities become unreliable at such FeO contents: in tests, LavAtmos returned an unphysical, K-dominated vapour of about $10^{10}$ bar at 39 wt% FeO, and anomalous behaviour already at 25 wt%. We therefore cap FeO at `FEO_MAX` = 18 wt%, a Mars-like value that behaved well. Above about IW−1.7, only fO$_2$ changes and the melt stays fixed. These oxidised cases represent a metal-free mantle with fixed FeO.

In [ ]:
OXIDE_MM = {"SiO2": 60.084, "MgO": 40.304, "Al2O3": 101.961, "TiO2": 79.866, "FeO": 71.844,
            "CaO": 56.077, "Na2O": 61.979, "K2O": 94.196, "Fe2O3": 159.688}   # [g mol-1]

base_melt = {str(k): float(v) for k, v in pd.read_csv(BASE_MELT, header=None).values}
base_total = sum(base_melt.values())

def melt_with_feo(feo_wt):
    # Set FeO to feo_wt and rescale the others so that the total is unchanged
    others = {k: v for k, v in base_melt.items() if k != "FeO"}
    scale = (base_total - feo_wt) / sum(others.values())
    comp = {k: v * scale for k, v in others.items()}
    comp["FeO"] = feo_wt
    return comp

def x_oxide(comp, oxide):
    mol = {k: v / OXIDE_MM[k] for k, v in comp.items()}
    return mol[oxide] / sum(mol.values())

def feo_for_diw(d):
    # Melt FeO [wt%] in ideal equilibrium with pure Fe metal at ΔIW = d, capped at FEO_MAX
    x = 10 ** (d / 2)
    if x >= x_oxide(melt_with_feo(FEO_MAX), "FeO"):
        return FEO_MAX
    return brentq(lambda w: x_oxide(melt_with_feo(w), "FeO") - x, 1e-4, FEO_MAX)

feo_wt = {d: feo_for_diw(d) for d in DIW}
melts  = {d: melt_with_feo(feo_wt[d]) for d in DIW}
for d, c in melts.items():
    assert abs(sum(c.values()) - base_total) < 1e-9, "composition total not conserved"
    assert all(v >= 0 for v in c.values()), "negative oxide abundance"
    if feo_wt[d] < FEO_MAX:     # uncapped cases must reproduce the requested ΔIW
        assert abs(2 * np.log10(x_oxide(c, "FeO")) - d) < 1e-6

melt_df = pd.DataFrame(melts).T
melt_df.index.name = "dIW"
melt_df["x_FeO"] = [x_oxide(melts[d], "FeO") for d in DIW]
melt_df["capped"] = [feo_wt[d] >= FEO_MAX for d in DIW]
melt_df.round(4)

## Vapour at imposed fO$_2$

The fO$_2$ is $\log_{10} f\mathrm{O_2} = \log_{10} f\mathrm{O_2}^\mathrm{IW,\,O'Neill}(T_\mathrm{melt}) + \Delta\mathrm{IW}$. The atmosphere is volatile-free: C, H, N, S and P are included only as traces. Results are cached as CSV files in `OUT_DIR`, so the notebook can be re-run without LavAtmos. As a check, we compare the O$_2$ partial pressure that LavAtmos returns with the imposed fO$_2$.

In [ ]:
iw_buffer = OxygenFugacity(model="oneill")   # O'Neill & Eggins (2002) IW buffer, as implemented in CALLIOPE
log_fO2_IW = iw_buffer(T_MELT)
print(f"log10 fO2(IW, O'Neill, {T_MELT:.0f} K) = {log_fO2_IW:.3f}")

lava = {}
for d in DIW:
    cache = os.path.join(OUT_DIR, f"lavatmos_diw{d:+.1f}_T{T_MELT:.0f}.csv")
    fO2 = 10 ** (log_fO2_IW + d)
    if os.path.exists(cache):
        dat = pd.read_csv(cache, index_col=0).iloc[:, 0]
        resid = float(dat.pop("MB_residual"))
        row = dat
        print(f"IW{d:+d}: loaded from cache")
    else:
        t0 = time.time()
        system = lavatmos3.melt_vapor_system(LavaPaths(os.path.join(OUT_DIR, "lavatmos_work", f"diw{d:+.1f}")))
        with in_dir(LAVA_DIR):
            row, resid = vaporise_fixed_fO2(system, T_MELT, melts[d], fO2)
        pd.concat([row, pd.Series({"MB_residual": resid})]).to_csv(cache)
        print(f"IW{d:+d}: {time.time()-t0:5.1f} s")
    lava[d] = (row, resid)
    assert abs(np.log10(float(row["O2"])) - np.log10(fO2)) < 0.05, "returned pO2 does not match the imposed fO2"

### From FastChem species to AGNI gases

FastChem labels species in Hill-like notation (`Fe1O1`, `O1Si1`, and so on). We map them to AGNI gas names, drop ions and electrons, and discard any gas with VMR below `VMR_FLOOR`. The fraction of the pressure that is dropped is reported. The total surface pressure is the sum of the retained neutral partial pressures.

In [ ]:
FC_TO_AGNI = {"Na": "Na", "K": "K", "Fe": "Fe", "Mg": "Mg", "Si": "Si", "Ca": "Ca", "Al": "Al", "Ti": "Ti",
              "O": "O", "O2": "O2", "Fe1O1": "FeO", "O1Si1": "SiO", "O2Si1": "SiO2", "Mg1O1": "MgO",
              "O1Ti1": "TiO", "O2Ti1": "TiO2", "Ca1O1": "CaO", "Al1O1": "AlO", "Na1O1": "NaO",
              "Mg2": "Mg2", "Na2": "Na2"}

def to_agni(row):
    neutral = row[[c for c in row.index if not c.endswith(("+", "-"))]]
    neutral = neutral[neutral > 0]
    pp = {FC_TO_AGNI[k]: float(v) for k, v in neutral.items() if k in FC_TO_AGNI}
    p_mapped = sum(pp.values())
    vmr = {g: p / p_mapped for g, p in pp.items() if p / p_mapped >= VMR_FLOOR}
    norm = sum(vmr.values())
    vmr = {g: v / norm for g, v in vmr.items()}
    return vmr, p_mapped * norm, 1.0 - p_mapped * norm / neutral.sum()

agni_in, surf = {}, []
for d in DIW:
    row, resid = lava[d]
    vmr, psurf, lost = to_agni(row)
    assert abs(sum(vmr.values()) - 1.0) < 1e-12
    agni_in[d] = (vmr, psurf)
    surf.append(dict(dIW=d, FeO_wt=feo_wt[d], p_surf_bar=psurf, frac_pressure_dropped=lost,
                     log_fO2=np.log10(float(row["O2"])), O_mass_balance_residual=resid,
                     x_Fe=vmr.get("Fe", 0.0), x_FeO=vmr.get("FeO", 0.0), x_SiO=vmr.get("SiO", 0.0),
                     x_Na=vmr.get("Na", 0.0), x_O2=vmr.get("O2", 0.0)))
surf_df = pd.DataFrame(surf).set_index("dIW")
case_label = lambda d: f"IW{d:+g} ({feo_wt[d]:.1f} wt% FeO)"
surf_df

Now we plot the surface partial pressures against ΔIW. Colours mark species identity, and each line is also labelled directly. The eight most abundant species (ranked across all cases) are plotted.

The second panel shows the oxygen mass-balance residual. Positive values mean the vapour carries less oxygen than congruent vaporisation would release, so the buffering mantle must take up the difference.

In [ ]:
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
DIW_RAMP = ["#86b6ef", "#6da7ec", "#3987e5", "#2a78d6", "#1c5cab", "#184f95", "#0d366b"]   # ordinal blue, reduced -> oxidised
diw_col = {d: DIW_RAMP[i] for i, d in enumerate(DIW)}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e4e3dd", "grid.linewidth": 0.6, "lines.linewidth": 2})
from matplotlib.ticker import ScalarFormatter, NullFormatter

def plain_log_ticks(ax, ticks):
    # Readable tick labels on a log x-axis (no overlapping minor-tick labels)
    ax.set_xticks(ticks)
    ax.xaxis.set_major_formatter(ScalarFormatter()); ax.xaxis.set_minor_formatter(NullFormatter())

def spread_labels(y, min_dlog=0.45):
    # Nudge label positions apart in log10 space so that end-of-line labels don't collide
    order = np.argsort(y)
    out = np.log10(np.asarray(y, dtype=float))
    for j in range(1, len(order)):
        out[order[j]] = max(out[order[j]], out[order[j - 1]] + min_dlog)
    return 10 ** out

pp_tab = pd.DataFrame({d: {g: v * agni_in[d][1] for g, v in agni_in[d][0].items()} for d in DIW}).T.fillna(0)
top = pp_tab.max().sort_values(ascending=False).index[:8]

fig, axs = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw=dict(width_ratios=[1.5, 1]))
ax = axs[0]
for i, g in enumerate(top):
    ax.plot(pp_tab.index, pp_tab[g], color=CAT[i], marker="o", ms=4)
ax.plot(surf_df.index, surf_df.p_surf_bar, color="#3d3d3a", ls=":", lw=1.5)
names = list(top) + ["total"]
ends = [max(pp_tab[g].iloc[-1], 1e-12) for g in top] + [surf_df.p_surf_bar.iloc[-1]]
for g, y_end, y_lab in zip(names, ends, spread_labels(ends)):
    ax.annotate(g, (DIW[-1], y_end), xytext=(DIW[-1] + 0.15, y_lab), textcoords="data",
                va="center", fontsize=9, color="#3d3d3a")
ax.set_xlim(DIW[0] - 0.2, DIW[-1] + 0.9)
ax.set_yscale("log"); ax.set_ylim(bottom=1e-6)
ax.set_xlabel("ΔIW (O'Neill)"); ax.set_ylabel("Surface partial pressure [bar]")
sec = ax.secondary_xaxis("top"); sec.set_xticks(DIW); sec.set_xticklabels([f"{feo_wt[d]:.3g}" for d in DIW])
sec.set_xlabel("Melt FeO [wt%]")
ax.set_title(f"Rock vapour above the melt at {T_MELT:.0f} K", loc="left")

ax = axs[1]
ax.plot(surf_df.index, surf_df.O_mass_balance_residual, color="#2a78d6", marker="o", ms=5)
ax.axhline(0, color="#3d3d3a", lw=1)
ax.set_yscale("symlog", linthresh=1e-3)
ax.set_xlabel("ΔIW (O'Neill)"); ax.set_ylabel("O mass-balance residual [bar]")
ax.set_title("Oxygen taken up by the mantle", loc="left")
fig.tight_layout(); save_fig(fig, "surface_vapour_vs_dIW"); plt.show()

## AGNI via `juliacall`

This follows the calling pattern in PROTEUS (`src/proteus/atmos_clim/agni.py`). In Python, the `!` of Julia's mutating functions is written `_b`. The aerosol table is built directly in Julia as a `Dict{String,Any}`, so that AGNI's validation sees native Julia types.

In [ ]:
from juliacall import Main as jl
jl.seval("using Pkg")
jl.Pkg.activate(AGNI_DIR)
jl.seval("import AGNI")
jl.AGNI.setup_logging(os.path.join(OUT_DIR, "agni.log"), 1)

AER_JL = jl.seval('Dict{String,Any}("FeO" => Dict{String,Any}(' + ", ".join(
    f'"{k}" => {repr(v) if isinstance(v, float) else chr(34) + v + chr(34)}' for k, v in AEROSOL.items()) + "))")
print(AER_JL)

`run_agni` runs one stage for one composition.

* In `prescribed` mode, `solve_prescribed` sets T(p) and computes the fluxes. For the `cond` and `aer` stages, we then apply FeO rainout to that fixed structure (`calc_composition!` with `rainout=true`), set the aerosol profile from the condensate yield (`set_aerosols!`), and recompute the fluxes.
* In `energy` mode, `solve_energy` handles rainout and aerosols internally at every iteration.

Both modes use `sol_type=1` (fixed surface temperature).

In [ ]:
def make_atmos(vmr, psurf, condense, aerosol):
    # Set up and allocate an Atmos_t for one composition; FeO may condense, and may form Mie aerosols
    atmos = jl.AGNI.atmosphere.Atmos_t()
    kw = dict(flag_aerosol=True, aerosol_species=AER_JL) if aerosol else dict(flag_aerosol=False)
    ok = jl.AGNI.atmosphere.setup_b(atmos, AGNI_DIR, OUT_DIR, SPFILE,
                                    INSTELLATION, S0_FACT, ALBEDO_B, ZENITH, T_MELT,
                                    GRAVITY, RADIUS, NLEV, psurf, P_TOP,
                                    jl.convert(jl.Dict, vmr), "",
                                    condensates=jl.convert(jl.Array[jl.String, 1], ["FeO"] if condense else []),
                                    flag_rayleigh=True, tmp_floor=50.0, check_integrity=False, **kw)
    assert bool(ok), "setup! failed"
    assert bool(jl.AGNI.atmosphere.allocate_b(atmos, STAR)), "allocate! failed"
    return atmos

def run_agni(tag, vmr, psurf, condense, aerosol, init_nc=None):
    # Run one stage with fixed T_surf = T_MELT, write a NetCDF file, and return diagnostics as numpy arrays
    atmos = make_atmos(vmr, psurf, condense, aerosol)
    t0 = time.time()
    try:
        if SOLVER == "prescribed":
            conv = bool(jl.AGNI.solver.solve_prescribed_b(atmos, sol_type=1, atm_type=ATM_TYPE))
            if condense:
                jl.AGNI.chemistry.calc_composition_b(atmos, False, False, True)   # oceans, chem, rainout
            if aerosol:
                jl.AGNI.atmosphere.set_aerosols_b(atmos)
            jl.AGNI.energy.calc_fluxes_b(atmos, radiative=True, convective=True, sens_heat=True)
        else:
            if init_nc is None:
                jl.AGNI.setpt.isothermal_b(atmos, 0.8 * T_MELT)
            else:
                jl.AGNI.setpt.fromncdf_b(atmos, init_nc)
            conv = bool(jl.AGNI.solver.solve_energy_b(atmos, sol_type=1, chem=False, rainout=condense,
                                                      latent=False, convect=True, sens_heat=True,
                                                      conduct=False, oceans=False, max_steps=MAX_STEPS,
                                                      easy_start=False, save_frames=False, modplot=0))
    except Exception as e:     # Julia-side exception, treat as failed
        print(f"    {tag}: solver raised {type(e).__name__}: {str(e)[:120]}")
        conv = False

    path = os.path.join(OUT_DIR, f"{tag}.nc")
    jl.AGNI.save.write_ncdf(atmos, path)
    gas = lambda g: np.array(atmos.gas_vmr[g]) if g in atmos.gas_names else np.zeros(NLEV)
    out = dict(tag=tag, converged=conv, runtime=time.time() - t0, nc=path,
               p=np.array(atmos.p), pl=np.array(atmos.pl), tmp=np.array(atmos.tmp), tmpl=np.array(atmos.tmpl),
               flux_n=np.array(atmos.flux_n), flux_tot=np.array(atmos.flux_tot),
               flux_u_lw=np.array(atmos.flux_u_lw), flux_u_sw=np.array(atmos.flux_u_sw),
               flux_d_sw=np.array(atmos.flux_d_sw),
               band_u_lw=np.array(atmos.band_u_lw), band_u_sw=np.array(atmos.band_u_sw),
               band_d_sw=np.array(atmos.band_d_sw),
               bands_min=np.array(atmos.bands_min), bands_max=np.array(atmos.bands_max),
               x_FeO=gas("FeO"), x_Fe=gas("Fe"), layer_mass=np.array(atmos.layer_σ),
               rainout=condense, coldtrap=bool(atmos.coldtrap),
               evap_efficiency=float(atmos.evap_efficiency), cloud_alpha=float(atmos.cloud_alpha),
               cond_yield=np.array(atmos.cond_yield["FeO"]) if condense else np.zeros(NLEV),
               aer_mmr=np.array(atmos.aerosol_arr_l["feo"]) if aerosol else np.zeros(NLEV))
    jl.AGNI.atmosphere.deallocate_b(atmos)
    print(f"    {tag:22s} ok={conv!s:5s} {out['runtime']:6.1f} s   T=[{out['tmp'].min():.0f}, {out['tmp'].max():.0f}] K   "
          f"FeO yield={out['cond_yield'].sum():.2e} kg m-2")
    return out

## Sweep

Each ΔIW is run through the three stages. In `energy` mode, runs that fail to converge are kept but flagged (hollow markers in the summary, and in the checks table).

In [ ]:
runs = {}
prev_base = None
for d in DIW:
    vmr, psurf = agni_in[d]
    print(f"{case_label(d)}   p_surf = {psurf:.3e} bar")
    stem = f"diw{d:+.1f}"
    r_base = run_agni(stem + "_base", vmr, psurf, condense=False, aerosol=False, init_nc=prev_base)
    r_cond = run_agni(stem + "_cond", vmr, psurf, condense=True,  aerosol=False, init_nc=r_base["nc"])
    r_aer  = run_agni(stem + "_aer",  vmr, psurf, condense=True,  aerosol=True,  init_nc=r_cond["nc"])
    runs[d] = dict(base=r_base, cond=r_cond, aer=r_aer)
    if r_base["converged"]:
        prev_base = r_base["nc"]

## Is this rainout condensation?

**Yes, in the `cond` and `aer` stages.** Those stages use `rainout=true` with FeO as a condensate. At every level AGNI compares $p_\mathrm{FeO} = x_\mathrm{FeO}\,p$ with $p_\mathrm{sat}(T)$. The treatment is as follows:

* **Condensation.** A supersaturated level has its FeO VMR reset to the saturation value, and the excess becomes condensate (`cond_yield`, kg m$^{-2}$).
* **Cold trap.** Levels above a condensing level cannot hold more FeO than the saturated VMR below them (`coldtrap=true`), so the gas above the cloud is depleted.
* **No re-evaporation.** `evap_efficiency = 0`, so the condensate rains out of the column rather than re-evaporating in drier layers beneath. Gas below the cloud keeps its surface VMR.
* **Aerosols.** In the `aer` stage, a fraction `cloud_alpha` of the local condensate yield is kept in the layer as suspended wüstite particles, $\mathrm{MMR}_i = \alpha\, m_{\mathrm{cond},i}/\sigma_i$. The rest is removed.

**The `base` stage has no condensation**, so its FeO can be supersaturated; it shows where clouds *would* form. The table reads the settings back from the AGNI structs, rather than relying on the values we passed in.

In [ ]:
rain_rows = []
for d in DIW:
    for k in ("base", "cond", "aer"):
        r = runs[d][k]
        lay = np.nonzero(r["cond_yield"] > 0)[0]
        rain_rows.append(dict(dIW=d, stage=k, rainout=r["rainout"], coldtrap=r["coldtrap"],
                              evap_efficiency=r["evap_efficiency"], cloud_alpha=r["cloud_alpha"],
                              n_condensing_layers=len(lay),
                              p_cloud_Pa=(f"{r['p'][lay.min()]:.3g}–{r['p'][lay.max()]:.3g}" if len(lay) else "-"),
                              x_FeO_top=r["x_FeO"][0], x_FeO_surface=r["x_FeO"][-1],
                              yield_kg_m2=r["cond_yield"].sum(),
                              aerosol_col_kg_m2=float(np.sum(r["aer_mmr"] * r["layer_mass"]))))
rain_df = pd.DataFrame(rain_rows).set_index(["dIW", "stage"])
rain_df

## Temperature structure and where clouds form

Each panel is one redox state. The **FeO dew-point curve** (dash-dot) is the temperature at which the surface FeO VMR would just saturate at each pressure, $p_\mathrm{sat}(T_\mathrm{dew}) = x_\mathrm{FeO}^\mathrm{surf}\,p$. FeO condenses wherever the profile lies to the *left* of (colder than) this curve.

Shaded bands mark condensing layers: blue for the `cond` stage and orange for aerosol-bearing layers in the `aer` stage. Grey crosses mark levels where the `base` profile is supersaturated.

In `prescribed` mode, all three stages share the same T(p). The FeO saturation curve comes from the same thermodynamic file that AGNI uses (`res/thermodynamics/FeO.nc`, `sat_P` in log$_{10}$ Pa).

In [ ]:
with nc.Dataset(os.path.join(AGNI_DIR, "res", "thermodynamics", "FeO.nc")) as ds:
    _satT, _satP = np.array(ds["sat_T"][:]), np.array(ds["sat_P"][:])
assert np.all(np.diff(_satP) >= 0), "saturation curve must be monotonic to invert it"
psat_FeO  = lambda T: 10.0 ** np.interp(T, _satT, _satP)            # [Pa]
T_dew_FeO = lambda pp: np.interp(np.log10(pp), _satP, _satT)        # partial pressure [Pa] -> [K]
sat_ratio = lambda r: r["x_FeO"] * r["p"] / psat_FeO(r["tmp"])

STAGE_STYLE = {"base": dict(color="#898781", ls=":"), "cond": dict(color="#2a78d6", ls="--"),
               "aer": dict(color="#eb6834", ls="-")}
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
ncol = 4
nrow = int(np.ceil((len(DIW) + 1) / ncol))
fig, axs = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 3.6 * nrow), sharey=True, sharex=True)
axs = list(axs.flat)
for ax, d in zip(axs, DIW):
    rb = runs[d]["base"]
    ax.plot(T_dew_FeO(rb["x_FeO"][-1] * rb["p"]), rb["p"] / 1e5, color="#3d3d3a", ls="-.", lw=1.2)
    for k in ("base", "cond", "aer"):
        r = runs[d][k]
        ax.plot(r["tmp"], r["p"] / 1e5, lw=1.8, alpha=1.0 if r["converged"] else 0.35, **STAGE_STYLE[k])
    for k, col in (("cond", "#b7d3f6"), ("aer", "#f6c9b3")):
        r = runs[d][k]
        mask = r["aer_mmr"] > 0 if k == "aer" else r["cond_yield"] > 0
        for i in np.nonzero(mask)[0]:
            ax.axhspan(r["pl"][i] / 1e5, r["pl"][i + 1] / 1e5, color=col, alpha=0.6, lw=0, zorder=0)
    sup = sat_ratio(rb) > 1
    ax.plot(rb["tmp"][sup], rb["p"][sup] / 1e5, "x", color="#b4b2a9", ms=3, mew=0.8)
    ax.set_title(case_label(d), loc="left", fontsize=9)
axs[len(DIW)].axis("off")
axs[len(DIW)].legend(handles=[Line2D([], [], **STAGE_STYLE["base"], label="base: no condensation"),
                              Line2D([], [], **STAGE_STYLE["cond"], label="cond: FeO rainout"),
                              Line2D([], [], **STAGE_STYLE["aer"], label="aer: rainout + aerosol opacity"),
                              Line2D([], [], color="#3d3d3a", ls="-.", lw=1.2, label="FeO dew point"),
                              Line2D([], [], color="#b4b2a9", ls="", marker="x", label="base supersaturated"),
                              Patch(color="#b7d3f6", label="condensing layer (cond)"),
                              Patch(color="#f6c9b3", label="aerosol layer (aer)")],
                     loc="center left", frameon=False, fontsize=8)
for ax in axs[len(DIW) + 1:]:
    ax.axis("off")
axs[0].set_yscale("log"); axs[0].invert_yaxis()
for i, ax in enumerate(axs[:len(DIW)]):
    if i % ncol == 0:
        ax.set_ylabel("Pressure [bar]")
    if i >= len(DIW) - ncol:
        ax.set_xlabel("Temperature [K]")
    ax.set_xlim(min(1000, 0.9 * T_skin), T_MELT + 300)
fig.suptitle(f"T(p) and FeO condensation ({SOLVER}), T$_\\mathrm{{surf}}$ = {T_MELT:.0f} K", x=0.01, ha="left")
fig.tight_layout(); save_fig(fig, "tp_profiles_clouds"); plt.show()

Next, the gas-phase FeO and the wüstite aerosol profiles for every redox state (`aer` stage). The gas VMR drops sharply at the cloud base because of the cold trap.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4.5), sharey=True)
for d in DIW:
    r = runs[d]["aer"]
    axs[0].plot(np.maximum(r["x_FeO"], 1e-30), r["p"] / 1e5, color=diw_col[d], label=case_label(d))
    axs[1].plot(np.maximum(r["aer_mmr"], 1e-30), r["p"] / 1e5, color=diw_col[d])
axs[0].set_xscale("log"); axs[0].set_xlim(1e-14, 1); axs[0].set_xlabel("FeO gas VMR")
axs[1].set_xscale("log"); axs[1].set_xlim(1e-9, 1e-1); axs[1].set_xlabel("FeO aerosol MMR [kg/kg]")
axs[0].set_yscale("log"); axs[0].invert_yaxis(); axs[0].set_ylabel("Pressure [bar]")
axs[0].legend(frameon=False, fontsize=7.5)
axs[0].set_title("Gas-phase FeO (aer)", loc="left"); axs[1].set_title("Wüstite aerosol (aer)", loc="left")
fig.tight_layout(); save_fig(fig, "feo_gas_and_aerosol_profiles"); plt.show()

## Summary against redox state

* The **OLR** is the upward longwave flux at the top of the atmosphere.
* The **Bond albedo** is the upward divided by the downward shortwave flux at the top.
* The **aerosol column** is $\sum_i \mathrm{MMR}_i\,\sigma_i$, where $\sigma_i$ is the layer column mass.
* The **cloud pressure range** covers the condensing layers in the `aer` stage.

Hollow markers indicate that the run did not converge. The `base` and `cond` curves coincide in the flux panels: they differ only in removing FeO gas, which has no opacity in `Volenfell/128`.

As an order-of-magnitude check on the aerosol effect: for 0.5 µm particles, the mass extinction coefficient is roughly $3Q_\mathrm{ext}/(4
ho r_\mathrm{eff}) pprox 500$ m$^2$ kg$^{-1}$ (taking $Q_\mathrm{ext}pprox2$ and $
ho = 5970$ kg m$^{-3}$). The aerosol column therefore corresponds to a visible optical depth of about $500	imes$ the column in kg m$^{-2}$, which is printed below.

In [ ]:
def summary(r):
    lay = np.nonzero(r["cond_yield"] > 0)[0]
    return dict(OLR=r["flux_u_lw"][0], albedo=r["flux_u_sw"][0] / r["flux_d_sw"][0],
                aer_col=float(np.sum(r["aer_mmr"] * r["layer_mass"])), yield_col=float(np.sum(r["cond_yield"])),
                p_cloud_top=r["p"][lay.min()] if len(lay) else np.nan,
                p_cloud_base=r["p"][lay.max()] if len(lay) else np.nan, converged=r["converged"])

summ = pd.DataFrame([dict(dIW=d, stage=k, **summary(runs[d][k])) for d in DIW for k in ("base", "cond", "aer")]
                    ).set_index(["dIW", "stage"])

fig, axs = plt.subplots(2, 2, figsize=(10, 7), sharex=True)
panels = [("OLR", "OLR [MW m$^{-2}$]", 1e6), ("albedo", "Bond albedo", 1.0), ("aer_col", "Aerosol column [kg m$^{-2}$]", 1.0)]
for ax, (key, lab, sc) in zip(list(axs.flat)[:3], panels):
    for k, ls, col in [("base", ":", "#898781"), ("cond", "--", "#1c5cab"), ("aer", "-", "#eb6834")]:
        sub = summ.xs(k, level="stage")
        yv = sub[key] / sc
        conv = sub["converged"].values.astype(bool)
        ax.plot(sub.index, yv, ls=ls, color=col, label=k)
        ax.plot(sub.index[conv], yv[conv], "o", ms=5, color=col)
        ax.plot(sub.index[~conv], yv[~conv], "o", ms=7, mfc="none", color=col)
    ax.set_ylabel(lab)
axs[0, 0].legend(frameon=False, fontsize=8)
ax = axs[1, 1]
sub = summ.xs("aer", level="stage")
ax.fill_between(sub.index, sub.p_cloud_top / 1e5, sub.p_cloud_base / 1e5, color="#f6c9b3", lw=0)
ax.plot(sub.index, sub.p_cloud_top / 1e5, color="#eb6834", marker="o", ms=4, label="cloud top")
ax.plot(sub.index, sub.p_cloud_base / 1e5, color="#eb6834", ls="--", marker="o", ms=4, label="cloud base")
ax.plot(surf_df.index, surf_df.p_surf_bar, color="#3d3d3a", ls=":", lw=1.5, label="surface")
ax.set_yscale("log"); ax.invert_yaxis(); ax.set_ylabel("Pressure [bar]"); ax.legend(frameon=False, fontsize=8)
for ax in axs[1]:
    ax.set_xlabel("ΔIW (O'Neill)")
fig.suptitle(f"Radiative effect of FeO clouds ({SOLVER})", x=0.01, ha="left")
fig.tight_layout(); save_fig(fig, "summary_vs_dIW"); plt.show()
summ["tau_est"] = 500.0 * summ["aer_col"]     # order-of-magnitude optical depth, see text
summ.round(5)

## Emission and reflectance spectra

These are the band-resolved fluxes at the top of the atmosphere on the 128 `Volenfell` bands.

* **Emission** is the upward longwave (thermal) flux per unit wavelength.
* **Reflectance** is the ratio of upward to downward shortwave flux in each band. It is shown only where the stellar input is non-negligible.

The upper panels show how the spectra change with redox state in the `aer` stage (the full calculation). The lower panels isolate the radiative effect of the FeO aerosols: the emission ratio and the reflectance difference between the calculation with aerosol opacity (`aer`) and the one without (`cond`). A difference is used for reflectance because the cloud-free reflectance is close to zero in some bands.

In [ ]:
def toa_spectra(r):
    wl = 0.5 * (r["bands_min"] + r["bands_max"]) * 1e6       # [µm]
    dwl = (r["bands_max"] - r["bands_min"]) * 1e6
    emis = r["band_u_lw"][0, :] / dwl                        # [W m-2 µm-1]
    down = r["band_d_sw"][0, :]
    refl = np.where(down > 1e-6 * down.max(), r["band_u_sw"][0, :] / np.maximum(down, 1e-300), np.nan)
    order = np.argsort(wl)
    return wl[order], emis[order], refl[order]

fig, axs = plt.subplots(2, 2, figsize=(12, 7.5), sharex="col", gridspec_kw=dict(height_ratios=[2, 1]))
for d in DIW:
    wl, e_aer, r_aer = toa_spectra(runs[d]["aer"])
    _, e_cond, r_cond = toa_spectra(runs[d]["cond"])
    c = diw_col[d]
    axs[0, 0].step(wl, e_aer, where="mid", color=c, lw=1.4, label=case_label(d))
    axs[0, 1].step(wl, r_aer, where="mid", color=c, lw=1.4)
    axs[1, 0].step(wl, e_aer / e_cond, where="mid", color=c, lw=1.4)
    axs[1, 1].step(wl, r_aer - r_cond, where="mid", color=c, lw=1.4)
axs[0, 0].set_yscale("log"); axs[0, 0].set_ylabel(r"Emission at TOA [W m$^{-2}$ µm$^{-1}$]")
axs[0, 0].set_ylim(bottom=axs[0, 0].get_ylim()[1] * 1e-6)
axs[0, 0].set_title("Thermal emission (aer)", loc="left")
axs[0, 1].set_ylabel("Reflectance (up / down SW)"); axs[0, 1].set_title("Reflectance (aer)", loc="left")
axs[1, 0].axhline(1, color="#3d3d3a", lw=1); axs[1, 1].axhline(0, color="#3d3d3a", lw=1)
axs[1, 0].set_ylabel("Emission aer / cond"); axs[1, 1].set_ylabel("Reflectance aer − cond")
for ax in axs[:, 0]:
    ax.set_xscale("log"); ax.set_xlim(0.3, 100); plain_log_ticks(ax, [0.3, 1, 3, 10, 30, 100])
for ax in axs[:, 1]:
    ax.set_xscale("log"); ax.set_xlim(0.2, 5); plain_log_ticks(ax, [0.2, 0.5, 1, 2, 5])
axs[1, 0].set_xlabel("Wavelength [µm]"); axs[1, 1].set_xlabel("Wavelength [µm]")
axs[0, 0].legend(frameon=False, fontsize=7.5, title="Redox state", title_fontsize=8)
fig.tight_layout(); save_fig(fig, "spectra_aerosol_effect"); plt.show()

The effect of redox state alone is shown relative to the IW−2 case, using the `aer` stage. Emission is shown as a ratio on a logarithmic axis, because at short wavelengths it spans many orders of magnitude between cases. Reflectance is shown as a difference.

In [ ]:
D_REF = -2
fig, axs = plt.subplots(1, 2, figsize=(12, 3.8))
wl0, e0, r0 = toa_spectra(runs[D_REF]["aer"])
for d in DIW:
    if d == D_REF:
        continue
    wl, e, r = toa_spectra(runs[d]["aer"])
    axs[0].step(wl, e / e0, where="mid", color=diw_col[d], lw=1.4, label=case_label(d))
    axs[1].step(wl, r - r0, where="mid", color=diw_col[d], lw=1.4)
axs[0].set_yscale("log"); axs[0].set_ylim(1e-2, 1e2); axs[0].axhline(1, color="#3d3d3a", lw=1)
axs[1].axhline(0, color="#3d3d3a", lw=1)
for ax in axs:
    ax.set_xscale("log"); ax.set_xlabel("Wavelength [µm]")
axs[0].set_xlim(0.3, 100); axs[1].set_xlim(0.2, 5)
plain_log_ticks(axs[0], [0.3, 1, 3, 10, 30, 100]); plain_log_ticks(axs[1], [0.2, 0.5, 1, 2, 5])
axs[0].set_ylabel(f"Emission / IW{D_REF:+d}"); axs[1].set_ylabel(f"Reflectance − IW{D_REF:+d}")
axs[0].legend(frameon=False, fontsize=7.5)
fig.tight_layout(); save_fig(fig, "spectra_relative_to_reference"); plt.show()

## Cross-check: are the converged states unique? (`energy` mode only)

A converged run shows only that the solver found *an* equilibrium. As an independent test, we reload each stored profile and recompute the fluxes under all three physics set-ups, without iterating. A profile is an equilibrium of a given set-up if the total flux is uniform with height. In an earlier sweep with congruent vaporisation, the aerosol-bearing solution at the highest FeO content ended on a different, cloud-free branch that was also an equilibrium of the aerosol-free physics.

This check is skipped in `prescribed` mode, because a prescribed structure is not an equilibrium.

In [ ]:
def flux_spread(nc_path, vmr, psurf, condense, aerosol):
    # Recompute fluxes on a stored T(p) under the given physics, without iterating
    atmos = make_atmos(vmr, psurf, condense, aerosol)
    jl.AGNI.setpt.fromncdf_b(atmos, nc_path)
    jl.AGNI.chemistry.calc_composition_b(atmos, False, False, condense)
    if aerosol:
        jl.AGNI.atmosphere.set_aerosols_b(atmos)
    jl.AGNI.energy.calc_fluxes_b(atmos, radiative=True, convective=True, sens_heat=True)
    ft = np.array(atmos.flux_tot)
    aer = float(np.max(np.array(atmos.aerosol_arr_l["feo"]))) if aerosol else 0.0
    jl.AGNI.atmosphere.deallocate_b(atmos)
    return (ft.max() - ft.min()) / np.abs(ft).max(), aer

xcheck = pd.DataFrame()
if SOLVER == "energy":
    xrows = []
    for d in DIW:
        if runs[d]["cond"]["cond_yield"].sum() <= 0 and runs[d]["aer"]["cond_yield"].sum() <= 0:
            continue     # nothing condensed, so the three set-ups are identical for this case
        vmr, psurf = agni_in[d]
        for prof in ("base", "cond", "aer"):
            for phys, (cd, ae) in {"dry": (False, False), "rainout": (True, False), "rainout+aerosol": (True, True)}.items():
                s, a = flux_spread(runs[d][prof]["nc"], vmr, psurf, cd, ae)
                xrows.append(dict(dIW=d, profile=prof, physics=phys, flux_spread_pct=100 * s, aer_mmr_max=a))
    xcheck = pd.DataFrame(xrows).set_index(["dIW", "profile", "physics"]) if xrows else xcheck
else:
    print("Skipped: SOLVER = 'prescribed'")
xcheck

## Physical checks

These are necessary conditions, not a validation of the physics.

* The temperature is positive and finite.
* VMRs sum to 1 at the surface.
* Fluxes are finite.
* In `energy` mode only, converged runs have a net flux that is uniform with height to within 1%.
* The aerosol mass is zero where aerosols are off and non-negative elsewhere.
* FeO is not supersaturated ($S \le 1+10^{-6}$) wherever rainout is on.
* The LavAtmos O$_2$ partial pressure matches the imposed fO$_2$ (asserted above).

In [ ]:
chk = []
for d in DIW:
    vmr, _ = agni_in[d]
    for k in ("base", "cond", "aer"):
        r = runs[d][k]
        fn = r["flux_n"]
        spread = (fn.max() - fn.min()) / max(abs(fn).max(), 1e-30)
        S = sat_ratio(r)
        chk.append(dict(dIW=d, stage=k, converged=r["converged"],
                        T_positive=bool(np.all(np.isfinite(r["tmpl"])) and np.all(r["tmpl"] > 0)),
                        vmr_sum_1=abs(sum(vmr.values()) - 1) < 1e-12,
                        flux_finite=bool(np.all(np.isfinite(r["flux_tot"]))),
                        flux_n_spread=spread,
                        energy_ok=(spread < 1e-2) if (SOLVER == "energy" and r["converged"]) else np.nan,
                        aer_mass_ok=bool(np.all(r["aer_mmr"] >= 0)) and (k == "aer" or np.all(r["aer_mmr"] == 0)),
                        max_S=float(S.max()),
                        no_supersat=(S.max() <= 1 + 1e-6) if k != "base" else np.nan))
chk_df = pd.DataFrame(chk).set_index(["dIW", "stage"])
hard = ["T_positive", "vmr_sum_1", "flux_finite", "aer_mass_ok"]
print("Hard checks all pass:", bool(chk_df[hard].all().all()))
print("No supersaturation where rainout is on:", bool(chk_df.no_supersat.dropna().astype(bool).all()))
print("Solver reported success:", int(chk_df.converged.sum()), "/", len(chk_df))
if SOLVER == "energy":
    print("Converged runs conserving energy:", bool(chk_df.energy_ok.dropna().astype(bool).all()))
chk_df

## Caveats and limitations

* **Prescribed structure.** In `prescribed` mode, T(p) is imposed (a dry adiabat to the skin temperature, then isothermal), and clouds and aerosols do not feed back on it. The results locate where FeO can condense on that structure, and give the instantaneous radiative effect of the clouds. They are not an equilibrium climate; use `SOLVER = "energy"` for that.
* **FeO gas has no opacity in `Volenfell/128`.** The spectral file includes the absorbers O$_2$, SiO, TiO, MgO, Na, K and Fe, plus the O$_2$–O$_2$ continuum. FeO(g) therefore affects the result only through its mass and condensation. The only radiative carriers of iron are Fe(g) and the wüstite aerosol.
* **The aerosol treatment is parametric.** The aerosol mass is `cloud_alpha` × condensate yield per layer (AGNI default `cloud_alpha = 0.01`). The size distribution is fixed (r$_\mathrm{eff}$ = 0.5 µm, σ$_g$ = 1.65), and there is no sedimentation or microphysics.
* **Latent heat is excluded** (`latent=false`). AGNI has no latent-heat data for FeO in any case.
* **The atmosphere is well mixed.** The surface VMRs from LavAtmos apply to the whole column, and only condensation modifies them; there is no thermochemistry aloft (`chem=false`).
* **Redox is imposed.** The mantle is treated as an infinite oxygen buffer, and the oxygen mass-balance residual shows how much oxygen it would have to exchange. Melt FeO comes from an ideal core–mantle relation and is capped at 18 wt%. Above about IW−1.7, the melt composition is fixed and only fO$_2$ varies. None of the melts include Fe$_2$O$_3$.
* **MELTS limitations.** MELTS activities are used at 2500 K for a fully liquid melt. At FeO contents above about 20 wt%, LavAtmos returned unphysical vapours in tests.
* **The planet is a single 1D column** with a fixed surface temperature (`sol_type=1`). The surface flux required to hold that temperature is not constrained.